In [1]:
import molscene
#scene = molscene.Scene.from_cif('6x63.cif')
#coords = scene[(scene['name']=='CB') | ((scene['name']=='CA') & (scene['resName']=='GLY'))][['x','y','z']].values[:,:3].astype(float)


In [2]:
import pandas as pd
def extract_atom_section(cif_file):
    """
    Extracts only the _atom section from an mmCIF file.

    Args:
        cif_file (str): Path to the CIF file.

    Returns:
        list: List of parsed atom data rows.
    """
    atom_data = []
    atom_header = []
    in_atom_section = False

    with open(cif_file, 'r') as file:
        for line in file:
            line = line.strip()
            
            # Skip empty lines and comments
            if not line or line.startswith("#"):
                continue
            
            # Detect the start of the _atom section
            if line.startswith("loop_"):
                in_atom_section = False  # Reset section flag
            
            elif line.startswith("_atom_site."):
                atom_header.append(line.split('.')[-1])
                in_atom_section = True  # Found relevant section, start collecting
            
            elif in_atom_section:
                atom_data.append(line)

    return atom_header,atom_data

# Example usage:
cif_file_path = "6x63.cif"
atom_header,atoms = extract_atom_section(cif_file_path)

atom_df = pd.DataFrame((line.split() for line in atoms),columns=atom_header)
atom_df.to_csv('6x63.csv',index=False)

In [3]:
import pandas as pd

import re
from concurrent.futures import ThreadPoolExecutor

pattern = re.compile(r'\s+')

def split_regex(line):
    return pattern.split(line)

def split_line(line):
    return line.split()

def extract_atom_section(cif_file):
    """
    Extracts only the _atom section from an mmCIF file.

    Args:
        cif_file (str): Path to the CIF file.

    Returns:
        list: List of parsed atom data rows.
    """
    atom_data = []
    atom_header = []
    in_atom_section = False

    with open(cif_file, 'r') as file:
        for line in file:
            line = line.strip()
            
            # Skip empty lines and comments
            if not line or line.startswith("#"):
                continue
            
            # Detect the start of the _atom section
            if line.startswith("loop_"):
                in_atom_section = False  # Reset section flag
            
            elif line.startswith("_atom_site."):
                atom_header.append(line.split('.')[-1])
                in_atom_section = True  # Found relevant section, start collecting
            
            elif in_atom_section:
                atom_data.append(line)

    return atom_header,atom_data

# Example usage:
cif_file_path = "6x63.cif"
atom_header,atoms = extract_atom_section(cif_file_path)

atom_df = pd.DataFrame((line.split() for line in atoms),columns=atom_header)
atom_df

#pd.DataFrame(map(split_line, atoms[:100]))
# with ThreadPoolExecutor() as executor:
#     split_lines = list(executor.map(split_line, atoms),chunksize=50000)
    
# atom_df = pd.Series(atoms)


,group_PDB,id,type_symbol,label_atom_id,label_alt_id,label_comp_id,label_asym_id,label_entity_id,label_seq_id,pdbx_PDB_ins_code,...,Cartn_y,Cartn_z,occupancy,B_iso_or_equiv,pdbx_formal_charge,auth_seq_id,auth_comp_id,auth_asym_id,auth_atom_id,pdbx_PDB_model_num
0,ATOM,1,N,N,.,PRO,A,1,1,?,...,136.652,61.325,1.00,0.00,?,1,PRO,A,N,1
1,ATOM,2,C,CA,.,PRO,A,1,1,?,...,137.859,62.207,1.00,0.00,?,1,PRO,A,CA,1
2,ATOM,3,C,C,.,PRO,A,1,1,?,...,138.705,61.790,1.00,0.00,?,1,PRO,A,C,1
3,ATOM,4,O,O,.,PRO,A,1,1,?,...,138.231,61.017,1.00,0.00,?,1,PRO,A,O,1
4,ATOM,5,C,CB,.,PRO,A,1,1,?,...,138.440,61.954,1.00,0.00,?,1,PRO,A,CB,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1359283,ATOM,1359284,H,HD12,.,LEU,NN,1,231,?,...,133.193,25.434,1.00,0.00,?,231,LEU,FF,HD12,1
1359284,ATOM,1359285,H,HD13,.,LEU,NN,1,231,?,...,133.011,27.061,1.00,0.00,?,231,LEU,FF,HD13,1
1359285,ATOM,1359286,H,HD21,.,LEU,NN,1,231,?,...,135.963,27.128,1.00,0.00,?,231,LEU,FF,HD21,1
1359286,ATOM,1359287,H,HD22,.,LEU,NN,1,231,?,...,135.088,28.508,1.00,0.00,?,231,LEU,FF,HD22,1


In [33]:
%%timeit
pd.DataFrame(map(split_line, atoms[:100000]))

344 ms ± 6.84 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [34]:
%%timeit
pd.DataFrame(line.split() for line in atoms[:100000])

349 ms ± 3.38 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [31]:
%%timeit
pd.DataFrame(map(split_regex, atoms[:10000]))

42.9 ms ± 1.51 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [4]:
import mmap
import pandas as pd

def extract_atom_section(cif_file):
    """
    Extracts only the _atom section from an mmCIF file as fast as possible.
    
    Reads the file using a memory map, decodes each line only once,
    uses partition() for header extraction, and limits splits to the expected number of fields.
    
    Args:
        cif_file (str): Path to the CIF file.
    
    Returns:
        tuple: (List of column headers, List of parsed atom data rows)
    """
    atom_header = []
    atom_data = []
    in_atom_section = False
    
    with open(cif_file, 'rb') as f:
        # Memory-map the file for faster access
        mm = mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ)
        
        # Iterate over each line (in bytes)
        for raw_line in iter(mm.readline, b""):
            # Decode and strip the line (do this only once)
            line = raw_line.decode('utf-8').strip()
            
            # Skip empty lines and comments
            if not line or line[0] == "#":
                continue

            # Reset section flag when encountering a new loop
            if line.startswith("loop_"):
                in_atom_section = False
            
            # Detect header lines in the _atom_site section
            elif line.startswith("_atom_site."):
                # Use partition() to avoid creating an extra list from split('.')
                _, _, col_name = line.partition(".")
                atom_header.append(col_name)
                in_atom_section = True
            
            # Once in the _atom section, split each data line
            elif in_atom_section:
                # If a new field begins (but not an atom field), exit atom section
                if line.startswith("_") and not line.startswith("_atom_site."):
                    break
                # Use maxsplit based on header length to avoid extra work
                atom_data.append(line.split(None, len(atom_header) - 1))
        
        mm.close()
    return atom_header, atom_data

# Example usage:
cif_file_path = "6x63.cif"
atom_header, atoms = extract_atom_section(cif_file_path)
atom_df = pd.DataFrame(atoms, columns=atom_header)
atom_df


,group_PDB,id,type_symbol,label_atom_id,label_alt_id,label_comp_id,label_asym_id,label_entity_id,label_seq_id,pdbx_PDB_ins_code,...,Cartn_y,Cartn_z,occupancy,B_iso_or_equiv,pdbx_formal_charge,auth_seq_id,auth_comp_id,auth_asym_id,auth_atom_id,pdbx_PDB_model_num
0,ATOM,1,N,N,.,PRO,A,1,1,?,...,136.652,61.325,1.00,0.00,?,1,PRO,A,N,1
1,ATOM,2,C,CA,.,PRO,A,1,1,?,...,137.859,62.207,1.00,0.00,?,1,PRO,A,CA,1
2,ATOM,3,C,C,.,PRO,A,1,1,?,...,138.705,61.790,1.00,0.00,?,1,PRO,A,C,1
3,ATOM,4,O,O,.,PRO,A,1,1,?,...,138.231,61.017,1.00,0.00,?,1,PRO,A,O,1
4,ATOM,5,C,CB,.,PRO,A,1,1,?,...,138.440,61.954,1.00,0.00,?,1,PRO,A,CB,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1359283,ATOM,1359284,H,HD12,.,LEU,NN,1,231,?,...,133.193,25.434,1.00,0.00,?,231,LEU,FF,HD12,1
1359284,ATOM,1359285,H,HD13,.,LEU,NN,1,231,?,...,133.011,27.061,1.00,0.00,?,231,LEU,FF,HD13,1
1359285,ATOM,1359286,H,HD21,.,LEU,NN,1,231,?,...,135.963,27.128,1.00,0.00,?,231,LEU,FF,HD21,1
1359286,ATOM,1359287,H,HD22,.,LEU,NN,1,231,?,...,135.088,28.508,1.00,0.00,?,231,LEU,FF,HD22,1


In [5]:
import mmap
import re
import pandas as pd

def tokenize_line(line):
    """
    Tokenizes an mmCIF line using a regex pattern instead of split(),
    preserving quoted fields and handling whitespace efficiently.
    """
    mmcif_re = re.compile(
        r"(?:"
        r"(?:_(.+?)[.](\S+))"               "|"  # _category.attribute
        r"(?:['](.*?)(?:[']\s|[']$))"       "|"  # single quoted strings
        r'(?:["](.*?)(?:["]\s|["]$))'       "|"  # double quoted strings             
        r"(?:\s*#.*$)"                      "|"  # comments (dumped)
        r"(\S+)"                                 # unquoted words
        r")")
    
    tokens = []
    for match in mmcif_re.finditer(line):
        tgroups = match.groups()
        if tgroups != (None, None, None, None, None):
            if tgroups[2] is not None:
                qs = tgroups[2]
            elif tgroups[3] is not None:
                qs = tgroups[3]
            else:
                qs = None
            tokens.append(qs if qs is not None else tgroups[4])
    return tokens

def extract_atom_section(cif_file):
    """
    Extracts only the _atom_site section from an mmCIF file efficiently.
    
    Uses memory-mapping, regex-based tokenization, and structured processing.
    
    Args:
        cif_file (str): Path to the CIF file.
    
    Returns:
        tuple: (List of column headers, List of parsed atom data rows)
    """
    atom_header = []
    atom_data = []
    in_atom_section = False
    
    with open(cif_file, 'rb') as f:
        # Memory-map the file for faster access
        mm = mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ)
        
        # Iterate over each line (in bytes)
        for raw_line in iter(mm.readline, b""):
            # Decode and strip the line (do this only once)
            line = raw_line.decode('utf-8').strip()
            
            # Skip empty lines and comments
            if not line or line[0] == "#":
                continue

            # Reset section flag when encountering a new loop
            if line.startswith("loop_"):
                in_atom_section = False
            
            # Detect header lines in the _atom_site section
            elif line.startswith("_atom_site."):
                _, _, col_name = line.partition(".")
                atom_header.append(col_name)
                in_atom_section = True
            
            # Once in the _atom section, process each data line
            elif in_atom_section:
                # If a new field begins (but not an atom field), exit atom section
                if line.startswith("_") and not line.startswith("_atom_site."):
                    break
                # Tokenize using regex instead of split()
                atom_data.append(tokenize_line(line))
        
        mm.close()
    return atom_header, atom_data

# Example usage:
cif_file_path = "6x63.cif"
atom_header, atoms = extract_atom_section(cif_file_path)
atom_df = pd.DataFrame(atoms, columns=atom_header)
atom_df


,group_PDB,id,type_symbol,label_atom_id,label_alt_id,label_comp_id,label_asym_id,label_entity_id,label_seq_id,pdbx_PDB_ins_code,...,Cartn_y,Cartn_z,occupancy,B_iso_or_equiv,pdbx_formal_charge,auth_seq_id,auth_comp_id,auth_asym_id,auth_atom_id,pdbx_PDB_model_num
0,ATOM,1,N,N,.,PRO,A,1,1,?,...,136.652,61.325,1.00,0.00,?,1,PRO,A,N,1
1,ATOM,2,C,CA,.,PRO,A,1,1,?,...,137.859,62.207,1.00,0.00,?,1,PRO,A,CA,1
2,ATOM,3,C,C,.,PRO,A,1,1,?,...,138.705,61.790,1.00,0.00,?,1,PRO,A,C,1
3,ATOM,4,O,O,.,PRO,A,1,1,?,...,138.231,61.017,1.00,0.00,?,1,PRO,A,O,1
4,ATOM,5,C,CB,.,PRO,A,1,1,?,...,138.440,61.954,1.00,0.00,?,1,PRO,A,CB,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1359283,ATOM,1359284,H,HD12,.,LEU,NN,1,231,?,...,133.193,25.434,1.00,0.00,?,231,LEU,FF,HD12,1
1359284,ATOM,1359285,H,HD13,.,LEU,NN,1,231,?,...,133.011,27.061,1.00,0.00,?,231,LEU,FF,HD13,1
1359285,ATOM,1359286,H,HD21,.,LEU,NN,1,231,?,...,135.963,27.128,1.00,0.00,?,231,LEU,FF,HD21,1
1359286,ATOM,1359287,H,HD22,.,LEU,NN,1,231,?,...,135.088,28.508,1.00,0.00,?,231,LEU,FF,HD22,1


In [ ]:
import mmap
import re
import pandas as pd
from multiprocessing import Pool, cpu_count

mmcif_re = re.compile(
    r"(?:"
    r"(?:_(.+?)[.](\S+))"               "|"  # _category.attribute
    r"(?:['](.*?)(?:[']\s|[']$))"       "|"  # single quoted strings
    r'(?:[\"](.*?)(?:[\"]\s|[\"]$))'    "|"  # double quoted strings             
    r"(?:\s*#.*$)"                      "|"  # comments (dumped)
    r"(\S+)"                                 # unquoted words
    r")")

def tokenize_line(line):
    """
    Tokenizes an mmCIF line using a regex pattern instead of split(),
    preserving quoted fields and handling whitespace efficiently.
    """

    
    tokens = []
    for match in mmcif_re.finditer(line):
        tgroups = match.groups()
        if tgroups != (None, None, None, None, None):
            if tgroups[2] is not None:
                qs = tgroups[2]
            elif tgroups[3] is not None:
                qs = tgroups[3]
            else:
                qs = None
            tokens.append(qs if qs is not None else tgroups[4])
    return line.split()

def process_lines(lines):
    """ Helper function to process a batch of lines in parallel. """
    return [line.split() for line in lines]

def extract_atom_section(cif_file):
    """
    Extracts only the _atom_site section from an mmCIF file efficiently.
    
    Uses memory-mapping, regex-based tokenization, structured processing,
    and parallel processing for faster data extraction.
    
    Args:
        cif_file (str): Path to the CIF file.
    
    Returns:
        tuple: (List of column headers, List of parsed atom data rows)
    """
    atom_header = []
    atom_data = []
    in_atom_section = False
    lines_to_process = []
    
    with open(cif_file, 'rb') as f:
        # Memory-map the file for faster access
        mm = mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ)
        
        # Iterate over each line (in bytes)
        for raw_line in iter(mm.readline, b""):
            # Decode and strip the line (do this only once)
            line = raw_line.decode('utf-8').strip()
            
            # Skip empty lines and comments
            if not line or line[0] == "#":
                continue

            # Reset section flag when encountering a new loop
            if line.startswith("loop_"):
                in_atom_section = False
            
            # Detect header lines in the _atom_site section
            elif line.startswith("_atom_site."):
                _, _, col_name = line.partition(".")
                atom_header.append(col_name)
                in_atom_section = True
            
            # Once in the _atom section, process each data line
            elif in_atom_section:
                # If a new field begins (but not an atom field), exit atom section
                if line.startswith("_") and not line.startswith("_atom_site."):
                    break
                # Collect lines for parallel processing
                lines_to_process.append(line)
        
        mm.close()
    
    # Use multiprocessing to tokenize lines in parallel
    num_workers = cpu_count()
    with Pool(num_workers) as pool:
        atom_data = pool.map(tokenize_line, lines_to_process)
    
    return atom_header, atom_data

# Example usage:
cif_file_path = "6x63.cif"
atom_header, atoms = extract_atom_section(cif_file_path)
atom_df = pd.DataFrame(atoms, columns=atom_header)
atom_df

ValueError: 21 columns passed, passed data had 127 columns

['', None, None, None, None, 'ATOM', ' ', None, None, None, None, '6', '']

In [ ]:
import mmap
import re
import numpy as np
import multiprocessing as mp
import cython
from cpython.bytes cimport PyBytes_AS_STRING
from libc.string cimport strchr

# Compile regex only once
mmcif_re = re.compile(
    r"(?:"
    r"(?:_(.+?)[.](\S+))"               "|"  # _category.attribute
    r"(?:['](.*?)(?:[']\s|[']$))"       "|"  # single quoted strings
    r'(?:[\"](.*?)(?:[\"]\s|[\"]$))' "|"  # double quoted strings             
    r"(?:\s*#.*$)"                      "|"  # comments (dumped)
    r"(\S+)"                                 # unquoted words
    r")")

@cython.boundscheck(False)
@cython.wraparound(False)
def tokenize_line(bytes line_bytes):
    """ Cython-optimized tokenizer to extract fields efficiently. """
    cdef const char *s = PyBytes_AS_STRING(line_bytes)
    cdef char *p
    cdef list tokens = []
    cdef char *start
    cdef int in_quote = 0
    cdef int in_token = 0
    cdef bytes token
    
    p = s
    while p[0] != b'\0':
        if p[0] in (b' ', b'\t', b'\n'):
            if in_token:
                token = line_bytes[start - s:p - s]
                tokens.append(token.decode('utf-8'))
                in_token = 0
            p += 1
        else:
            if not in_token:
                start = p
                in_token = 1
            p += 1
    if in_token:
        token = line_bytes[start - s:p - s]
        tokens.append(token.decode('utf-8'))
    return tokens

def process_chunk(lines):
    """Process a chunk of lines in parallel."""
    return [tokenize_line(line.encode('utf-8')) for line in lines]

def extract_atom_section(cif_file):
    """
    Extracts only the _atom_site section from an mmCIF file using parallel processing, Cython, and NumPy.
    
    Args:
        cif_file (str): Path to the CIF file.
    
    Returns:
        tuple: (List of column headers, NumPy array of parsed atom data)
    """
    atom_header = []
    atom_data = []
    in_atom_section = False
    lines_to_process = []
    
    with open(cif_file, 'rb') as f:
        mm = mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ)
        
        for raw_line in iter(mm.readline, b""):
            line = raw_line.decode('utf-8').strip()
            if not line or line[0] == "#":
                continue

            if line.startswith("loop_"):
                in_atom_section = False
            elif line.startswith("_atom_site."):
                _, _, col_name = line.partition(".")
                atom_header.append(col_name)
                in_atom_section = True
            elif in_atom_section:
                if line.startswith("_") and not line.startswith("_atom_site."):
                    break
                lines_to_process.append(line)
        mm.close()
    
    # Process lines in parallel using multiprocessing
    num_workers = mp.cpu_count()
    chunk_size = len(lines_to_process) // num_workers
    with mp.Pool(num_workers) as pool:
        results = pool.map(process_chunk, [lines_to_process[i::num_workers] for i in range(num_workers)])
    
    atom_data = np.array([item for sublist in results for item in sublist], dtype=object)
    
    return atom_header, atom_data

# Example usage:
cif_file_path = "6x63.cif"
atom_header, atoms = extract_atom_section(cif_file_path)
print(f"Headers: {atom_header}")
print(f"Atom Data Shape: {atoms.shape}")

SyntaxError: invalid syntax (3986914239.py, line 6)

In [ ]:
from

In [1]:
import pandas as pd
from molscene.pdbxparser import pdbxparser  # the compiled Cython module

cif_file_path = "6x63.cif"
atom_header, atoms = pdbxparser.extract_atom_section(cif_file_path)
atom_df = pd.DataFrame(atoms, columns=atom_header)
atom_df

,group_PDB,id,type_symbol,label_atom_id,label_alt_id,label_comp_id,label_asym_id,label_entity_id,label_seq_id,pdbx_PDB_ins_code,...,Cartn_y,Cartn_z,occupancy,B_iso_or_equiv,pdbx_formal_charge,auth_seq_id,auth_comp_id,auth_asym_id,auth_atom_id,pdbx_PDB_model_num
0,ATOM,1,N,N,.,PRO,A,1,1,?,...,136.652,61.325,1.00,0.00,?,1,PRO,A,N,1
1,ATOM,13,H,HG2,.,PRO,A,1,1,?,...,137.365,61.381,1.00,0.00,?,1,PRO,A,HG2,1
2,ATOM,25,H,H,.,ILE,A,1,2,?,...,140.294,62.954,1.00,0.00,?,2,ILE,A,H,1
3,ATOM,37,C,CA,.,VAL,A,1,3,?,...,140.848,58.308,1.00,0.00,?,3,VAL,A,CA,1
4,ATOM,49,H,HG21,.,VAL,A,1,3,?,...,138.413,55.824,1.00,0.00,?,3,VAL,A,HG21,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1359283,ATOM,1359240,H,H,.,ARG,NN,1,229,?,...,132.053,28.394,1.00,0.00,?,229,ARG,FF,H,1
1359284,ATOM,1359252,H,HH22,.,ARG,NN,1,229,?,...,139.665,24.134,1.00,0.00,?,229,ARG,FF,HH22,1
1359285,ATOM,1359264,H,HG12,.,VAL,NN,1,230,?,...,130.654,31.438,1.00,0.00,?,230,VAL,FF,HG12,1
1359286,ATOM,1359276,C,CD2,.,LEU,NN,1,231,?,...,135.731,27.627,1.00,0.00,?,231,LEU,FF,CD2,1


In [35]:
pwd

'/home/cb/Development/molscene'

<15423x15423 sparse matrix of type '<class 'numpy.float64'>'
	with 1767984 stored elements in COOrdinate format>

In [4]:
print('Hello')

Hello


In [32]:
import pandas as pd

def extract_atom_section(cif_file):
    """
    Extracts only the _atom section from an mmCIF file.

    Args:
        cif_file (str): Path to the CIF file.

    Returns:
        list: List of parsed atom data rows.
    """
    atom_data = []
    atom_header = []
    in_atom_section = False

    with open(cif_file, 'r') as file:
        for line in file:
            line = line.strip()
            
            # Skip empty lines and comments
            if not line or line.startswith("#"):
                continue
            
            # Detect the start of the _atom section
            if line.startswith("loop_"):
                in_atom_section = False  # Reset section flag
            
            elif line.startswith("_atom_site."):
                atom_header.append(line.split('.')[-1])
                in_atom_section = True  # Found relevant section, start collecting
            
            elif in_atom_section:
                # Collect atom data
                atom_data.append(line.split())

    return atom_header,atom_data

# Example usage:
cif_file_path = "6x63.cif"
atom_header,atoms = extract_atom_section(cif_file_path)
atom_df = pd.DataFrame(atoms, columns=atom_header)

atom_df

,group_PDB,id,type_symbol,label_atom_id,label_alt_id,label_comp_id,label_asym_id,label_entity_id,label_seq_id,pdbx_PDB_ins_code,...,Cartn_y,Cartn_z,occupancy,B_iso_or_equiv,pdbx_formal_charge,auth_seq_id,auth_comp_id,auth_asym_id,auth_atom_id,pdbx_PDB_model_num
0,ATOM,1,N,N,.,PRO,A,1,1,?,...,136.652,61.325,1.00,0.00,?,1,PRO,A,N,1
1,ATOM,2,C,CA,.,PRO,A,1,1,?,...,137.859,62.207,1.00,0.00,?,1,PRO,A,CA,1
2,ATOM,3,C,C,.,PRO,A,1,1,?,...,138.705,61.790,1.00,0.00,?,1,PRO,A,C,1
3,ATOM,4,O,O,.,PRO,A,1,1,?,...,138.231,61.017,1.00,0.00,?,1,PRO,A,O,1
4,ATOM,5,C,CB,.,PRO,A,1,1,?,...,138.440,61.954,1.00,0.00,?,1,PRO,A,CB,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1359283,ATOM,1359284,H,HD12,.,LEU,NN,1,231,?,...,133.193,25.434,1.00,0.00,?,231,LEU,FF,HD12,1
1359284,ATOM,1359285,H,HD13,.,LEU,NN,1,231,?,...,133.011,27.061,1.00,0.00,?,231,LEU,FF,HD13,1
1359285,ATOM,1359286,H,HD21,.,LEU,NN,1,231,?,...,135.963,27.128,1.00,0.00,?,231,LEU,FF,HD21,1
1359286,ATOM,1359287,H,HD22,.,LEU,NN,1,231,?,...,135.088,28.508,1.00,0.00,?,231,LEU,FF,HD22,1


In [26]:
len(atoms[1])

21

In [8]:
atom_header

['_atom_site.group_PDB',
 '_atom_site.id',
 '_atom_site.type_symbol',
 '_atom_site.label_atom_id',
 '_atom_site.label_alt_id',
 '_atom_site.label_comp_id',
 '_atom_site.label_asym_id',
 '_atom_site.label_entity_id',
 '_atom_site.label_seq_id',
 '_atom_site.pdbx_PDB_ins_code',
 '_atom_site.Cartn_x',
 '_atom_site.Cartn_y',
 '_atom_site.Cartn_z',
 '_atom_site.occupancy',
 '_atom_site.B_iso_or_equiv',
 '_atom_site.pdbx_formal_charge',
 '_atom_site.auth_seq_id',
 '_atom_site.auth_comp_id',
 '_atom_site.auth_asym_id',
 '_atom_site.auth_atom_id',
 '_atom_site.pdbx_PDB_model_num']

In [ ]:
dir(atoms)

In [ ]:
len(atoms)

In [ ]:
coords_charge = scene[(scene['name']=='CB') & (scene['resName'].isin(['ASP','GLU','LYS','ARG']))][['x','y','z']].values[:,:3].astype(float)


In [ ]:
coords.shape

In [ ]:
import numpy as np

def compute_distance_matrix(coords, threshold=8.0, dense_limit=1000):
    """
    Compute the pairwise Euclidean distance matrix for a given frame.
    
    For small molecules (n_atoms < dense_limit), a full dense distance matrix is computed using
    scipy.spatial.distance.cdist. For large molecules, a KD-tree is built and only those pairs 
    with a distance less than or equal to 'threshold' are computed and stored in a sparse COO matrix.
    
    Parameters
    ----------
    frame_index : int, optional
        Index of the frame to compute distances for (default is 0).
    threshold : float, optional
        The distance threshold. For large molecules, only pairs with distance <= threshold are stored.
        (default is 8.0 Angstroms)
    dense_limit : int, optional
        If the number of atoms is less than this limit, a full dense distance matrix is computed.
        Otherwise, a sparse matrix is returned. (default is 1000)
        
    Returns
    -------
    numpy.ndarray or scipy.sparse.coo_matrix
        The distance matrix for the specified frame. If the number of atoms is below dense_limit,
        a full (dense) numpy array of shape (N, N) is returned. Otherwise, a scipy.sparse COO matrix
        of shape (N, N) containing only the distances below the threshold is returned.
    """
    # Get coordinates for the given frame.
    coords
    N = coords.shape[0]
    
    if N < dense_limit:
        # For small systems, compute the full distance matrix.
        from scipy.spatial.distance import cdist
        dmat = cdist(coords, coords, metric='euclidean')
        return dmat
    else:
        # For large systems, use a KD-tree to compute only the distances within the threshold.
        from scipy.spatial import cKDTree
        from scipy.sparse import coo_matrix
        tree = cKDTree(coords)
        
        # Attempt to use the KD-tree's built-in sparse_distance_matrix with a COO output.
        try:
            spmat = tree.sparse_distance_matrix(tree, threshold, output_type='coo_matrix')
        except TypeError:
            # Fallback: If output_type is not supported, convert the dictionary output to COO.
            spmat_dict = tree.sparse_distance_matrix(tree, threshold)
            rows, cols, data = [], [], []
            for (i, j), d in spmat_dict.items():
                # Only store one triangle (i < j) to avoid duplicates.
                if i < j:
                    rows.append(i)
                    cols.append(j)
                    data.append(d)
            spmat = coo_matrix((data, (rows, cols)), shape=(N, N))
        
        # Make the matrix symmetric.
        spmat_sym = spmat + spmat.T
        return spmat_sym

dmat = compute_distance_matrix(coords, threshold=20.0, dense_limit=1000)
print(dmat)

In [ ]:
import scipy.sparse

# Calculate rho

rho = dmat.copy()
rho.data = 0.25 * (1 + np.tanh(5 * (dmat.data - 4.5))) * (1 + np.tanh(5 * (6.5 - dmat.data)))

rho.sum(axis=1).shape


In [ ]:
rho

In [ ]:
1767984/15423/15423

In [ ]:

        sequence_mask_rho = frustration.compute_mask(selected_matrix, 
                                                     maximum_contact_distance=None, 
                                                     minimum_sequence_separation = p.min_sequence_separation_rho)
        sequence_mask_contact = frustration.compute_mask(self.distance_matrix, 
                                                     maximum_contact_distance=p.distance_cutoff_contact, 
                                                     minimum_sequence_separation = p.min_sequence_separation_contact)
        
        self._decoy_fluctuation = {}
        self.minimally_frustrated_threshold=.78

        # Calculate rho
        rho = 0.25 
        rho *= (1 + np.tanh(p.eta * (selected_matrix- p.r_min)))
        rho *= (1 + np.tanh(p.eta * (p.r_max - selected_matrix)))
        rho *= sequence_mask_rho
        self.rho=rho
        
        #Calculate sigma water
        rho_r = (rho).sum(axis=1)
        if self.full_pdb_distance_matrix.shape!=self.distance_matrix.shape:
            if self.burial_in_context==True:
                self.init_index_shift=pdb_structure.init_index_shift
                self.fin_index_shift=pdb_structure.fin_index_shift
                rho_r=rho_r[self.init_index_shift:self.fin_index_shift]
        self.rho_r=rho_r
        rho_b = np.expand_dims(rho_r, 1)
        rho1 = np.expand_dims(rho_r, 0)
        rho2 = np.expand_dims(rho_r, 1)
        sigma_water = 0.25 * (1 - np.tanh(p.eta_sigma * (rho1 - p.rho_0))) * (1 - np.tanh(p.eta_sigma * (rho2 - p.rho_0)))
        sigma_protein = 1 - sigma_water

        #Calculate theta and indicators
        theta = 0.25 * (1 + np.tanh(p.eta * (self.distance_matrix - p.r_min))) * (1 + np.tanh(p.eta * (p.r_max - self.distance_matrix)))
        thetaII = 0.25 * (1 + np.tanh(p.eta * (self.distance_matrix - p.r_minII))) * (1 + np.tanh(p.eta * (p.r_maxII - self.distance_matrix)))
        burial_indicator = np.tanh(p.burial_kappa * (rho_b - p.burial_ro_min)) + np.tanh(p.burial_kappa * (p.burial_ro_max - rho_b))
        direct_indicator = theta[:, :, np.newaxis, np.newaxis]
        water_indicator = thetaII[:, :, np.newaxis, np.newaxis] * sigma_water[:, :, np.newaxis, np.newaxis]
        protein_indicator = thetaII[:, :, np.newaxis, np.newaxis] * sigma_protein[:, :, np.newaxis, np.newaxis]
        
        if expose_indicator_functions:
            self.indicators=[]
            self.indicators.append(burial_indicator[:,0])
            self.indicators.append(burial_indicator[:,1])
            self.indicators.append(burial_indicator[:,2])
            
            self.indicators.append(direct_indicator[:,:,0,0]*sequence_mask_contact)
            self.indicators.append(protein_indicator[:,:,0,0]*sequence_mask_contact)
            self.indicators.append(water_indicator[:,:,0,0]*sequence_mask_contact)
            
            self.gamma_array=[]
            temp_burial_gamma=self.burial_gamma[self.aa_map_awsem_list]
            temp_burial_gamma[0]=0
            temp_burial_gamma *= -0.5 * p.k_contact
            self.gamma_array.append(temp_burial_gamma[:,0])
            self.gamma_array.append(temp_burial_gamma[:,1])
            self.gamma_array.append(temp_burial_gamma[:,2])

            for contact_gamma in [self.direct_gamma, self.protein_gamma, self.water_gamma]:
                temp_gamma = contact_gamma[self.aa_map_awsem_x, self.aa_map_awsem_y].copy()
                temp_gamma[0, :] = 0
                temp_gamma[:, 0] = 0
                temp_gamma *= -0.5 * self.k_contact
                self.gamma_array.append(temp_gamma)

            self.burial_indicator = burial_indicator
            self.direct_indicator = direct_indicator
            self.water_indicator = water_indicator
            self.protein_indicator = protein_indicator
            

        J_index = np.meshgrid(range(self.N), range(self.N), range(self.q), range(self.q), indexing='ij', sparse=False)
        h_index = np.meshgrid(range(self.N), range(self.q), indexing='ij', sparse=False)

        #Burial energy
        burial_energy = 0.5 * p.k_contact * self.burial_gamma[h_index[1]] * burial_indicator[:, np.newaxis, :]
        self.burial_energy = burial_energy

        #Contact energy
        direct = direct_indicator * self.direct_gamma[J_index[2], J_index[3]]
        water_mediated = water_indicator * self.water_gamma[J_index[2], J_index[3]]
        protein_mediated = protein_indicator  * self.protein_gamma[J_index[2], J_index[3]]
        contact_energy = p.k_contact * np.array([direct, water_mediated, protein_mediated]) * sequence_mask_contact[np.newaxis, :, :, np.newaxis, np.newaxis]

        # Compute electrostatics
        if p.k_electrostatics!=0:
            self.sequence_cutoff=min(p.min_sequence_separation_electrostatics, p.min_sequence_separation_contact)
            self.distance_cutoff=None
            
            
            electrostatics_mask = frustration.compute_mask(self.distance_matrix, maximum_contact_distance=None, minimum_sequence_separation=p.min_sequence_separation_electrostatics)
            # ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']
            charges = np.array([0, 1, 0, -1, 0, 0, -1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0])
            charges2 = charges[:,np.newaxis]*charges[np.newaxis,:]

            electrostatics_indicator = 1 / (self.distance_matrix + 1E-6) * np.exp(-self.distance_matrix / p.electrostatics_screening_length) * electrostatics_mask
            electrostatics_energy = -p.k_electrostatics * (charges2[np.newaxis,np.newaxis,:,:]*electrostatics_indicator[:,:,np.newaxis,np.newaxis])

            contact_energy = np.append(contact_energy, electrostatics_energy[np.newaxis,:,:,:,:], axis=0)
            if expose_indicator_functions:
                self.indicators.append(electrostatics_indicator)
                temp_gamma=0.5 * p.k_electrostatics * charges2[self.aa_map_awsem_x, self.aa_map_awsem_y]
                temp_gamma[0,:]=0
                temp_gamma[:,0]=0
                self.gamma_array.append(temp_gamma)
        else:
            self.sequence_cutoff=p.min_sequence_separation_contact
            self.distance_cutoff=p.distance_cutoff_contact
        self.mask = frustration.compute_mask(self.distance_matrix, maximum_contact_distance=self.distance_cutoff, minimum_sequence_separation = self.sequence_cutoff)

        self.contact_energy = contact_energy

        # Compute fast properties
        self.aa_freq = frustration.compute_aa_freq(self.sequence)
        self.contact_freq = frustration.compute_contact_freq(self.sequence)
        self.potts_model = {}
        self.potts_model['h'] = burial_energy.sum(axis=-1)[:, self.aa_map_awsem_list]
        self.potts_model['J'] = contact_energy.sum(axis=0)[:, :, self.aa_map_awsem_x, self.aa_map_awsem_y]
        
        # Set the gap energy to zero
        self.potts_model['h'][:, 0] = 0
        self.potts_model['J'][:, :, 0, :] = 0
        self.potts_model['J'][:, :, :, 0] = 0
        self._native_energy=None

    def compute_configurational_decoy_statistics(self, n_decoys=4000,aa_freq=None):
        # ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']
        _AA='ARNDCQEGHILKMFPSTWYV'
        if aa_freq is None:
            seq_index = np.array([_AA.find(aa) for aa in self.sequence])
            N=self.N
        else:
            N=self.N*10
            total = sum(aa_freq)
            probabilities = [freq / total for freq in aa_freq.ravel()]
            seq_index = np.random.choice(a=len(aa_freq), size=N, p=probabilities)
        
        distances = np.triu(self.distance_matrix)
        distances = distances[(distances<self.distance_cutoff_contact) & (distances>0)]

        rho_b = np.expand_dims(self.rho_r, 1) #(n,1)
        rho1 = np.expand_dims(self.rho_r, 0) #(1,n)
        rho2 = np.expand_dims(self.rho_r, 1) #(n,1)

        sigma_water = 0.25 * (1 - np.tanh(self.eta_sigma * (rho1 - self.rho_0))) * (1 - np.tanh(self.eta_sigma * (rho2 - self.rho_0))) #(n,n)
        sigma_protein = 1 - sigma_water #(n,n)

        #Calculate theta and indicators
        theta = 0.25 * (1 + np.tanh(self.eta * (distances - self.r_min))) * (1 + np.tanh(self.eta * (self.r_max - distances))) # (c,)
        thetaII = 0.25 * (1 + np.tanh(self.eta * (distances - self.r_minII))) * (1 + np.tanh(self.eta * (self.r_maxII - distances))) #(c,)
        burial_indicator = np.tanh(self.burial_kappa * (rho_b - self.burial_ro_min)) + np.tanh(self.burial_kappa * (self.burial_ro_max - rho_b)) #(n,3)
           
        charges = np.array([0, 1, 0, -1, 0, 0, -1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0])
        electrostatics_indicator = np.exp(-distances / self.electrostatics_screening_length) / distances

        decoy_energies=np.zeros(n_decoys)
        #decoy_data=[None]*n_decoys
        #decoy_data_columns=['decoy_i','rand_i_resno','rand_j_resno','ires_type','jres_type','i_resno','j_resno','rij','rho_i','rho_j','water_energy','burial_energy_i','burial_energy_j','electrostatic_energy','tert_frust_decoy_energies']
        for i in range(n_decoys):
            c=np.random.randint(0,len(distances))
            n1=np.random.randint(0,self.N)
            n2=np.random.randint(0,self.N)
            qi1=np.random.randint(0,N)
            qi2=np.random.randint(0,N)
            q1=seq_index[qi1]
            q2=seq_index[qi2]

            
            burial_energy1 = (-0.5 * self.k_contact * self.burial_gamma[q1] * burial_indicator[n1]).sum(axis=0)
            burial_energy2 = (-0.5 * self.k_contact * self.burial_gamma[q2] * burial_indicator[n2]).sum(axis=0)
            
            direct = theta[c] * self.direct_gamma[q1, q2]
            water_mediated = sigma_water[n1,n2] * thetaII[c] * self.water_gamma[q1,q2]
            protein_mediated = sigma_protein[n1,n2] * thetaII[c] * self.protein_gamma[q1,q2]
            contact_energy = -self.k_contact * (direct+water_mediated+protein_mediated)
            electrostatics_energy = self.k_electrostatics * electrostatics_indicator[c]*charges[q1]*charges[q2]

            decoy_energies[i]=(burial_energy1+burial_energy2+contact_energy+electrostatics_energy)
            #decoy_data[i]=[i, qi1, qi2, q1, q2, n1, n2, distances[c], self.rho_r[n1], self.rho_r[n2], contact_energy/4.184, burial_energy1/4.184, burial_energy2/4.184, electrostatics_energy/4.184, decoy_energies[i]]
            
        mean_decoy_energy = np.mean(decoy_energies)
        std_decoy_energy = np.std(decoy_energies)
        return mean_decoy_energy, std_decoy_energy
    
    def compute_configurational_energies(self):
        _AA='ARNDCQEGHILKMFPSTWYV'
        seq_index = np.array([_AA.find(aa) for aa in self.sequence])
        distances = np.triu(self.distance_matrix)
        distances = distances[(distances<self.distance_cutoff_contact) & (distances>0)]
        n_contacts=len(distances)

        n = self.distance_matrix.shape[0]  # Assuming self.distance_matrix is defined and square
        tri_upper_indices = np.triu_indices(n, k=1)  # k=1 excludes the diagonal
        valid_pairs = (self.distance_matrix[tri_upper_indices] < self.distance_cutoff_contact) & \
                      (self.distance_matrix[tri_upper_indices] > 0)
        indices1,indices2 = (tri_upper_indices[0][valid_pairs], tri_upper_indices[1][valid_pairs])

        # for n1,n2,c in zip(indices1,indices2,range(n_contacts)):
        #     assert self.distance_matrix[n1,n2] == distances[c]
        
        rho_b = np.expand_dims(self.rho_r, 1) #(n,1)
        rho1 = np.expand_dims(self.rho_r, 0) #(1,n)
        rho2 = np.expand_dims(self.rho_r, 1) #(n,1)

        sigma_water = 0.25 * (1 - np.tanh(self.eta_sigma * (rho1 - self.rho_0))) * (1 - np.tanh(self.eta_sigma * (rho2 - self.rho_0))) #(n,n)
        sigma_protein = 1 - sigma_water #(n,n)

        #Calculate theta and indicators
        theta = 0.25 * (1 + np.tanh(self.eta * (distances - self.r_min))) * (1 + np.tanh(self.eta * (self.r_max - distances))) # (c,)
        thetaII = 0.25 * (1 + np.tanh(self.eta * (distances - self.r_minII))) * (1 + np.tanh(self.eta * (self.r_maxII - distances))) #(c,)
        burial_indicator = np.tanh(self.burial_kappa * (rho_b - self.burial_ro_min)) + np.tanh(self.burial_kappa * (self.burial_ro_max - rho_b)) #(n,3)
           
        charges = np.array([0, 1, 0, -1, 0, 0, -1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0])
        electrostatics_indicator = np.exp(-distances / self.electrostatics_screening_length) / distances

        # decoy_data_columns=['decoy_i','i_resno','j_resno','ires_type','jres_type','aa1','aa2','rij','rho_i','rho_j','water_energy','burial_energy_i','burial_energy_j','electrostatic_energy','total_energies']
        # decoy_data=[]
        configurational_energies=np.zeros((n,n))
        for c in range(n_contacts):
            n1=indices1[c]
            n2=indices2[c]
            q1=seq_index[n1]
            q2=seq_index[n2]

            burial_energy1 = (-0.5 * self.k_contact * self.burial_gamma[q1] * burial_indicator[n1]).sum(axis=0)
            burial_energy2 = (-0.5 * self.k_contact * self.burial_gamma[q2] * burial_indicator[n2]).sum(axis=0)
            
            direct = theta[c] * self.direct_gamma[q1, q2]
            water_mediated = sigma_water[n1,n2] * thetaII[c] * self.water_gamma[q1,q2]
            protein_mediated = sigma_protein[n1,n2] * thetaII[c] * self.protein_gamma[q1,q2]
            contact_energy = -self.k_contact * (direct+water_mediated+protein_mediated)
            electrostatics_energy = self.k_electrostatics * electrostatics_indicator[c]*charges[q1]*charges[q2]

            energy=(burial_energy1+burial_energy2+contact_energy+electrostatics_energy)
            configurational_energies[n1,n2]=energy
            configurational_energies[n2,n1]=energy
            # decoy_data+=[[c, n1, n2, q1, q2, _AA[q1],_AA[q2], distances[c], self.rho_r[n1], self.rho_r[n2], contact_energy/4.184, burial_energy1/4.184, burial_energy2/4.184, electrostatics_energy/4.184, energy/4.184]]
        # import pandas as pd
        return configurational_energies #, pd.DataFrame(decoy_data, columns=decoy_data_columns)
    
    def configurational_frustration(self,aa_freq=None, correction=0, n_decoys=4000):
        mean_decoy_energy, std_decoy_energy = self.compute_configurational_decoy_statistics(n_decoys=n_decoys,aa_freq=aa_freq)
        return -(self.compute_configurational_energies()-mean_decoy_energy)/(std_decoy_energy+correction)

In [ ]:
((scene['name']=='CA') & (scene['resName']=='GLY'))

In [ ]:
import seaborn as sns
sns.heatmap(dmat.toarray(), cmap='viridis')

In [ ]:
dmat

# Using `molscene.Scene`: A Quickstart Guide

This notebook demonstrates how to use the `Scene` class from the `molscene` package for molecular coordinate manipulation and visualization.

In [1]:
# Import required libraries
import numpy as np
from molscene import Scene

## Creating a Scene from Coordinates

You can create a `Scene` from a NumPy array of coordinates. Each row is an atom or point, and columns are x, y, z.

In [2]:
# Example coordinates: two points
coords = np.array([[0, 0, 0], [1, 1, 1]])
scene = Scene(coords)
scene

,x,y,z,chainID,resSeq,iCode,altLoc,model,name,element,occupancy,tempFactor,resName,chain_index,res_index,atom_index
0,0,0,0,A,1,,,1,P000,C,1.0,1.0,,0,0,0
1,1,1,1,A,1,,,1,P001,C,1.0,1.0,,0,0,1


## Accessing and Modifying Scene Metadata

You can attach metadata to a Scene, such as an author or description.

In [3]:
scene.author = "Alice"
scene.description = "A simple two-point scene."
print(f"Author: {scene.author}")
print(f"Description: {scene.description}")

Author: Alice
Description: A simple two-point scene.


## Operator Overloading: Addition and Subtraction

You can add or subtract vectors to translate the scene, or add two scenes to concatenate them.

In [4]:
# Translate scene by a vector
translated = scene + [1, 2, 3]
print(translated.get_coordinates())

     x    y    z
0  1.0  2.0  3.0
1  2.0  3.0  4.0


In [5]:
# Concatenate two scenes
scene2 = Scene([[2, 2, 2]])
merged = scene + scene2
print(merged.get_coordinates())

   x  y  z
0  0  0  0
1  1  1  1
2  2  2  2


## Multiplication, Division, and Negation

You can scale or invert coordinates using arithmetic operators.

In [6]:
# Scale coordinates
scaled = scene * 2
print(scaled.get_coordinates())

# Divide coordinates
divided = scene / 2
print(divided.get_coordinates())

# Negate coordinates
negated = -scene
print(negated.get_coordinates())

     x    y    z
0  0.0  0.0  0.0
1  2.0  2.0  2.0
     x    y    z
0  0.0  0.0  0.0
1  0.5  0.5  0.5
   x  y  z
0  0  0  0
1 -1 -1 -1


## Working with Multiple Coordinate Frames

A Scene can store multiple coordinate frames (e.g., for molecular dynamics).

In [7]:
# Add a second frame translated by [10, 20, 30]
frames = np.stack([
    scene.get_coordinates().to_numpy(),
    scene.get_coordinates().to_numpy() + np.array([10, 20, 30])
], axis=0)
scene.set_coordinate_frames(frames)
print("All frames:\n", scene.get_coordinate_frames())

All frames:
 [[[ 0  0  0]
  [ 1  1  1]]

 [[10 20 30]
  [11 21 31]]]


## Using Scene with NumPy Arrays

You can easily convert Scene coordinates to and from NumPy arrays for further analysis.

In [8]:
# Get coordinates as a NumPy array
coords_np = scene.get_coordinates().to_numpy()
print(coords_np)

[[0 0 0]
 [1 1 1]]


In [ ]:
import numpy as np
import pandas as pd
from molscene.Scene import Scene
data = pd.DataFrame({
    'name': ['CA', 'CB', 'FE', 'O', 'N', 'CA', 'C'],
    'resname': ['ALA', 'GLY', 'HEM', 'HOH', 'ALA', 'CYS', 'CYS'],
    'chain': ['A', 'A', 'A', 'A', 'A', 'A', 'A'],
    'resSeq': [10, 11, 12, 13, 14, 15, 16],
    'x': np.arange(7),
    'y': np.arange(7),
    'z': np.arange(7),
    'mass': [12, 13, 56, 16, 14, 12, 12],
})
scene = Scene(data)

In [13]:
ca = scene.select("not protein")
ca

,name,resName,chainID,resSeq,x,y,z,mass,chain,resid,iCode,altloc,model,element,occupancy,beta,resname,chain_index,res_index,atom_index
0,CA,ALA,A,10,0,0,0,12,A,1,,,1,C,1.0,1.0,,0,0,0
1,CB,GLY,A,11,1,1,1,13,A,1,,,1,C,1.0,1.0,,0,0,1
2,FE,HEM,A,12,2,2,2,56,A,1,,,1,C,1.0,1.0,,0,0,2
3,O,HOH,A,13,3,3,3,16,A,1,,,1,C,1.0,1.0,,0,0,3
4,N,ALA,A,14,4,4,4,14,A,1,,,1,C,1.0,1.0,,0,0,4
5,CA,CYS,A,15,5,5,5,12,A,1,,,1,C,1.0,1.0,,0,0,5
6,C,CYS,A,16,6,6,6,12,A,1,,,1,C,1.0,1.0,,0,0,6
